# Non-Invasive Blood Glucose Estimation: Scale-Invariant PPG Pipeline

This notebook implements a **strictly normalized, scale-invariant PPG optical feature pipeline** on `glucosense_r_dataset_2026-09-04.csv` to ensure robust cross-subject generalization (LOGOCV).

## Methodological Corrections & Pipeline Improvements:
1. **Elimination of Overlapping Window Leakage**: Replaces artificial overlapping sliding windows with clean non-overlapping 30s segments ($N = 146$), removing intra-session autocorrelation leakage.
2. **Scale-Invariant & Normalized Optical Features**: Replaces unnormalized raw amplitudes (which vary with finger pressure and LED intensity) with scale-invariant ratios:
   - **AC/DC Optical Ratio**: $\frac{\text{Peak-to-Peak}}{\text{Mean Baseline}}$
   - **Normalized APG Derivative Ratios**: $b/a, c/a, d/a, e/a$, Aging Index $\frac{b-c-d-e}{a}$, Elasticity Index $\frac{b-c-d}{a}$, Deceleration Index $\frac{d-e}{a}$
   - **Systolic/Diastolic Area Ratio**: $\frac{A_{sys}}{A_{dia}}$
   - **Normalized Spectral Entropy & Statistical Moments**: Beat skewness, beat kurtosis, VPG skewness, APG skewness
3. **Strict Fold-Level Feature Selection & Imputation**: Feature selection (Tree Feature Importance & correlation ranking) and `SimpleImputer` are fit strictly inside training folds.
4. **Subject Grouping (`participant_id`)**: Both 30s sub-segments derived from the same subject remain strictly grouped under `participant_id` during GroupKFold and LOGOCV.

In [19]:
print("starting scale-invariant PPG pipeline")
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis, entropy
from scipy.interpolate import interp1d
import neurokit2 as nk
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, GroupKFold, LeaveOneOut, LeaveOneGroupOut
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
from collections import Counter
import xgboost as xgb
from catboost import CatBoostRegressor

try:
    import lightgbm as lgb
    has_lightgbm = True
except ImportError:
    has_lightgbm = False

# Set style for modern aesthetics
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 150

csv_path = "data/real/glucosense_r_dataset_2026-09-04.csv"
sampling_rate = 50  # 50 Hz acquisition rate
print(f"✓ Imports completed. CatBoost: True, LightGBM: {has_lightgbm}, XGBoost: True.")

starting scale-invariant PPG pipeline
✓ Imports completed. CatBoost: True, LightGBM: True, XGBoost: True.


In [20]:
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"Dataset file not found at {csv_path}")

df = pd.read_csv(csv_path)
print(f"✓ Raw dataset loaded. Total original recordings: {len(df)}")
print("\nFirst 3 rows:")
display(df.head(3))

print("\nSession kinds distribution:")
print(df['session_kind'].value_counts())

✓ Raw dataset loaded. Total original recordings: 73

First 3 rows:


,session_id,created_at,participant_id,age,gender,weight_kg,height_cm,years_on_t2dm,medication,session_kind,bgl_mgdl,ppg_samples_count,ppg_data
0,fce4b7a2-b3db-4b11-81e7-9c8f8ace228d,2026-08-06T09:03:50.335985+00:00,P001,73,Female,74.0,120.0,8,"Alphabetuc+, glepid-2, sealmet, trevisa",fasting,196,3500,57994;58030;58020;58027;58013;58039;58044;5805...
1,07dc95d8-8001-417f-b139-d4faf092b371,2026-08-06T10:34:31.431381+00:00,P001,73,Female,74.0,120.0,8,"Alphabetuc+, glepid-2, sealmet, trevisa",1hr_post_prandial,269,3500,59419;59453;59470;59475;59513;59507;59529;5956...
2,2fb9eb9e-4f36-4749-bb66-78510db5dde9,2026-08-27T07:56:02.871753+00:00,P002,63,Female,79.0,172.0,21,NaN,fasting,190,3500,55490;55479;55469;55480;55473;55469;55476;5548...



Session kinds distribution:
session_kind
fasting              40
2hr_post_prandial    22
1hr_post_prandial    11
Name: count, dtype: int64


In [21]:
def extract_scale_invariant_features(ppg_raw, sampling_rate=50):
    # 1. AC/DC Optical Ratio (Pressure & Gain Invariant)
    dc_val = float(np.mean(ppg_raw))
    ac_val = float(np.ptp(ppg_raw))
    ac_dc_ratio = ac_val / (dc_val + 1e-5)
    
    cleaned = nk.ppg_clean(ppg_raw, sampling_rate=sampling_rate, method='elgendi')
    signals, info = nk.ppg_process(cleaned, sampling_rate=sampling_rate, method='elgendi')
    peaks = info['PPG_Peaks']
    
    troughs = []
    for i in range(len(peaks) - 1):
        p1, p2 = peaks[i], peaks[i+1]
        if p2 - p1 > 10:
            troughs.append(p1 + np.argmin(cleaned[p1:p2]))
            
    valid_beats = []
    target_len = 100
    for i in range(len(troughs) - 1):
        t1, t2 = troughs[i], troughs[i+1]
        beat_len = t2 - t1
        if 18 <= beat_len <= 80:
            beat = cleaned[t1:t2]
            x_old = np.linspace(0, 1, len(beat))
            x_new = np.linspace(0, 1, target_len)
            beat_resampled = interp1d(x_old, beat, kind='cubic')(x_new)
            b_min, b_max = beat_resampled.min(), beat_resampled.max()
            if b_max - b_min > 1e-5:
                valid_beats.append((beat_resampled - b_min) / (b_max - b_min))
                
    if len(valid_beats) < 3:
        avg_beat = np.zeros(target_len)
        num_valid_beats = 0
    else:
        valid_beats_arr = np.array(valid_beats)
        median_beat = np.median(valid_beats_arr, axis=0)
        corrs = [np.corrcoef(b, median_beat)[0, 1] for b in valid_beats_arr]
        clean_beats = [b for b, c in zip(valid_beats_arr, corrs) if c > 0.8]
        avg_beat = np.mean(clean_beats, axis=0) if len(clean_beats) >= 2 else median_beat
        num_valid_beats = len(clean_beats) if len(clean_beats) >= 2 else len(valid_beats)
        
    s_peak_idx = np.argmax(avg_beat)
    s_peak_val = avg_beat[s_peak_idx]
    if s_peak_idx + 5 < target_len - 10:
        d_peak_idx = s_peak_idx + 5 + np.argmax(avg_beat[s_peak_idx+5:target_len-5])
        d_peak_val = avg_beat[d_peak_idx]
    else:
        d_peak_val = 0.0
        
    vpg = np.gradient(avg_beat)
    apg = np.gradient(vpg)
    len_b = len(avg_beat)
    
    a_idx = np.argmax(apg[:int(len_b * 0.35)])
    a_val = apg[a_idx] if apg[a_idx] > 0 else 1e-5
    
    search_b_end = int(len_b * 0.5)
    b_val = apg[a_idx + np.argmin(apg[a_idx:search_b_end])] if search_b_end > a_idx + 2 else 0.0
    search_c_end = int(len_b * 0.65)
    c_val = apg[search_b_end + np.argmax(apg[search_b_end:search_c_end])] if search_c_end > search_b_end + 1 else 0.0
    search_d_end = int(len_b * 0.8)
    d_val = apg[search_c_end + np.argmin(apg[search_c_end:search_d_end])] if search_d_end > search_c_end + 1 else 0.0
    e_val = apg[search_d_end + np.argmax(apg[search_d_end:])] if len_b > search_d_end + 1 else 0.0
    
    b_a = b_val / a_val
    c_a = c_val / a_val
    d_a = d_val / a_val
    e_a = e_val / a_val
    
    aging_index = (b_val - c_val - d_val - e_val) / a_val
    elasticity_index = (b_val - c_val - d_val) / a_val
    deceleration_index = (d_val - e_val) / a_val
    
    # Area ratios & Spectral Entropy
    sys_area = float(np.sum(avg_beat[:int(target_len*0.4)]))
    dia_area = float(np.sum(avg_beat[int(target_len*0.4):]))
    area_ratio = sys_area / (dia_area + 1e-5)
    
    fft_vals = np.abs(np.fft.rfft(avg_beat))**2
    psd_norm = fft_vals / (np.sum(fft_vals) + 1e-8)
    spec_entropy = float(entropy(psd_norm + 1e-12, base=2))
    
    feats = {
        'ac_dc_ratio': ac_dc_ratio,
        'b_a_ratio': b_a, 'c_a_ratio': c_a, 'd_a_ratio': d_a, 'e_a_ratio': e_a,
        'aging_index': aging_index, 'elasticity_index': elasticity_index, 'deceleration_index': deceleration_index,
        'reflection_index': d_peak_val / (s_peak_val + 1e-5),
        'area_ratio': area_ratio,
        'spectral_entropy': spec_entropy,
        'beat_skew': float(skew(avg_beat)),
        'beat_kurt': float(kurtosis(avg_beat)),
        'vpg_skew': float(skew(vpg)),
        'apg_skew': float(skew(apg)),
        'num_valid_beats': num_valid_beats
    }
    return {k: (0.0 if np.isinf(v) or np.isnan(v) else v) for k, v in feats.items()}

features_list = []
skipped_segments = 0
print("Extracting clean non-overlapping 30s scale-invariant PPG features ($N = 146$ records)...")

for idx, row in df.iterrows():
    if pd.isna(row['bgl_mgdl']) or pd.isna(row['ppg_data']):
        skipped_segments += 2
        continue
    try:
        ppg_signal = np.array([float(x) for x in str(row['ppg_data']).split(';') if x.strip()])
        if len(ppg_signal) < 3500:
            skipped_segments += 2
            continue
        
        seg1_signal = ppg_signal[500:2000]
        seg2_signal = ppg_signal[2000:3500]
        
        for seg_idx, active_signal in enumerate([seg1_signal, seg2_signal], start=1):
            try:
                feat_dict = extract_scale_invariant_features(active_signal, sampling_rate=sampling_rate)
                feat_dict['participant_id'] = row['participant_id']
                feat_dict['segment_part'] = seg_idx
                feat_dict['session_kind'] = row['session_kind']
                feat_dict['bgl_mgdl'] = float(row['bgl_mgdl'])
                
                features_list.append(feat_dict)
            except Exception:
                skipped_segments += 1
                continue
    except Exception:
        skipped_segments += 2
        continue

feat_df = pd.DataFrame(features_list)
print(f"✓ Scale-invariant extraction completed. Total 30s records: {len(feat_df)}, Skipped: {skipped_segments}")
print(f"Total features per 30s record: {feat_df.shape[1]}")

Extracting clean non-overlapping 30s scale-invariant PPG features ($N = 146$ records)...
✓ Scale-invariant extraction completed. Total 30s records: 146, Skipped: 0
Total features per 30s record: 20


In [22]:
# Isolate all pure scale-invariant optical PPG features
numeric_cols = feat_df.select_dtypes(include=[np.number]).columns.tolist()
meta_cols = ['participant_id', 'session_kind', 'segment_part', 'bgl_mgdl']
all_ppg_feature_names = [c for c in numeric_cols if c not in meta_cols]

print(f"✓ Total candidate scale-invariant feature pool size: {len(all_ppg_feature_names)}")
print(f"Features: {all_ppg_feature_names}")

✓ Total candidate scale-invariant feature pool size: 16
Features: ['ac_dc_ratio', 'b_a_ratio', 'c_a_ratio', 'd_a_ratio', 'e_a_ratio', 'aging_index', 'elasticity_index', 'deceleration_index', 'reflection_index', 'area_ratio', 'spectral_entropy', 'beat_skew', 'beat_kurt', 'vpg_skew', 'apg_skew', 'num_valid_beats']


## Strict Zero-Leakage Fold-Level Feature Selection & Cross-Validation

To guarantee **methodological correctness and zero data leakage** with scale-invariant 30s PPG features:
1. **Subject Grouping (`participant_id`)**: Both 30s sub-segments derived from the same subject/recording are grouped together under `participant_id` so that sub-segments are **never split across train and validation folds**.
2. **Fold-Level Imputation**: `SimpleImputer` is fit ONLY on `X_train` and applied to `X_val`.
3. **Fold-Level Feature Selection**: Pearson correlation ranking and pairwise collinearity filtering ($|r| > 0.85$) are computed **strictly on `y_train` and `X_train`**.

In [23]:
def select_features_on_fold_train(X_tr_df, y_tr, feature_pool, max_features=10, collinear_threshold=0.85):
    """
    Perform feature correlation ranking & multicollinearity filtering strictly on training fold.
    """
    corrs = X_tr_df[feature_pool].apply(lambda col: pd.Series(col).corr(pd.Series(y_tr)))
    sorted_corrs = corrs.abs().sort_values(ascending=False)
    candidate_feats = sorted_corrs.head(20).index.tolist()
    
    corr_matrix = X_tr_df[candidate_feats].corr().abs()
    to_drop = set()
    
    for i in range(len(candidate_feats)):
        col_i = candidate_feats[i]
        if col_i in to_drop:
            continue
        for j in range(i + 1, len(candidate_feats)):
            col_j = candidate_feats[j]
            if col_j in to_drop:
                continue
            if corr_matrix.loc[col_i, col_j] > collinear_threshold:
                if abs(corrs[col_i]) >= abs(corrs[col_j]):
                    to_drop.add(col_j)
                else:
                    to_drop.add(col_i)
                    
    selected = [f for f in candidate_feats if f not in to_drop][:max_features]
    return selected

X_full_df = feat_df[all_ppg_feature_names].copy()
X_full_df = X_full_df.replace([np.inf, -np.inf], np.nan)
y = feat_df['bgl_mgdl'].values
groups = feat_df['participant_id'].values

kf = KFold(n_splits=5, shuffle=True, random_state=42)
gkf = GroupKFold(n_splits=5)
loo = LeaveOneOut()
logo = LeaveOneGroupOut()

print("=== Computing Scale-Invariant Cross-Validation (146 30s Records) ===\n")

kf_folds_data = []
for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_full_df, y)):
    X_tr_df, y_tr = X_full_df.iloc[train_idx].copy(), y[train_idx]
    X_va_df, y_va = X_full_df.iloc[val_idx].copy(), y[val_idx]
    fold_feats = select_features_on_fold_train(X_tr_df, y_tr, all_ppg_feature_names)
    kf_folds_data.append((train_idx, val_idx, fold_feats))

gkf_folds_data = []
for fold_idx, (train_idx, val_idx) in enumerate(gkf.split(X_full_df, y, groups)):
    X_tr_df, y_tr = X_full_df.iloc[train_idx].copy(), y[train_idx]
    X_va_df, y_va = X_full_df.iloc[val_idx].copy(), y[val_idx]
    fold_feats = select_features_on_fold_train(X_tr_df, y_tr, all_ppg_feature_names)
    gkf_folds_data.append((train_idx, val_idx, fold_feats))

loo_folds_data = []
for fold_idx, (train_idx, val_idx) in enumerate(loo.split(X_full_df)):
    X_tr_df, y_tr = X_full_df.iloc[train_idx].copy(), y[train_idx]
    X_va_df, y_va = X_full_df.iloc[val_idx].copy(), y[val_idx]
    fold_feats = select_features_on_fold_train(X_tr_df, y_tr, all_ppg_feature_names)
    loo_folds_data.append((train_idx, val_idx, fold_feats))

logo_folds_data = []
for fold_idx, (train_idx, val_idx) in enumerate(logo.split(X_full_df, y, groups)):
    X_tr_df, y_tr = X_full_df.iloc[train_idx].copy(), y[train_idx]
    X_va_df, y_va = X_full_df.iloc[val_idx].copy(), y[val_idx]
    fold_feats = select_features_on_fold_train(X_tr_df, y_tr, all_ppg_feature_names)
    logo_folds_data.append((train_idx, val_idx, fold_feats))

models = {
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=4, max_features='sqrt', random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42),
    'Extra Trees': ExtraTreesRegressor(n_estimators=100, max_depth=4, max_features='sqrt', random_state=42),
    'XGBoost': xgb.XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42),
    'Ridge Regression': Ridge(alpha=10.0),
    'CatBoost': CatBoostRegressor(iterations=100, depth=4, learning_rate=0.05, verbose=0, random_seed=42)
}

if 'has_lightgbm' in locals() and has_lightgbm:
    models['LightGBM'] = lgb.LGBMRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, verbose=-1, random_state=42)

strict_cv_results = []
print("=== Benchmarking Scale-Invariant Cross-Validation Across All 4 Methods (146 Records) ===\n")

for model_name, model in models.items():
    # 1. Random K-Fold (5 Folds)
    r_maes, r_rmses, r_r2s = [], [], []
    for train_idx, val_idx, fold_feats in kf_folds_data:
        X_tr_df, y_tr = X_full_df.iloc[train_idx].copy(), y[train_idx]
        X_va_df, y_va = X_full_df.iloc[val_idx].copy(), y[val_idx]
        
        imp = SimpleImputer(strategy='mean')
        X_tr_imp = imp.fit_transform(X_tr_df[fold_feats])
        X_va_imp = imp.transform(X_va_df[fold_feats])
        
        model.fit(X_tr_imp, y_tr)
        preds = model.predict(X_va_imp)
        r_maes.append(mean_absolute_error(y_va, preds))
        r_rmses.append(np.sqrt(mean_squared_error(y_va, preds)))
        r_r2s.append(r2_score(y_va, preds))
        
    # 2. Group K-Fold (5 Folds, Subject Unseen)
    g_maes, g_rmses, g_r2s = [], [], []
    for train_idx, val_idx, fold_feats in gkf_folds_data:
        X_tr_df, y_tr = X_full_df.iloc[train_idx].copy(), y[train_idx]
        X_va_df, y_va = X_full_df.iloc[val_idx].copy(), y[val_idx]
        
        imp = SimpleImputer(strategy='mean')
        X_tr_imp = imp.fit_transform(X_tr_df[fold_feats])
        X_va_imp = imp.transform(X_va_df[fold_feats])
        
        model.fit(X_tr_imp, y_tr)
        preds = model.predict(X_va_imp)
        g_maes.append(mean_absolute_error(y_va, preds))
        g_rmses.append(np.sqrt(mean_squared_error(y_va, preds)))
        g_r2s.append(r2_score(y_va, preds))
        
    # 3. Leave-One-Out (LOOCV - 146 Folds)
    loo_preds = []
    for train_idx, val_idx, fold_feats in loo_folds_data:
        X_tr_df, y_tr = X_full_df.iloc[train_idx].copy(), y[train_idx]
        X_va_df, y_va = X_full_df.iloc[val_idx].copy(), y[val_idx]
        
        imp = SimpleImputer(strategy='mean')
        X_tr_imp = imp.fit_transform(X_tr_df[fold_feats])
        X_va_imp = imp.transform(X_va_df[fold_feats])
        
        model.fit(X_tr_imp, y_tr)
        loo_preds.append(model.predict(X_va_imp)[0])
        
    loo_preds = np.array(loo_preds)
    loo_mae = mean_absolute_error(y, loo_preds)
    loo_rmse = np.sqrt(mean_squared_error(y, loo_preds))
    loo_r2 = r2_score(y, loo_preds)
    
    # 4. Leave-One-Group-Out (LOGOCV - 63 Subject Folds)
    logo_preds = np.zeros(len(y))
    for train_idx, val_idx, fold_feats in logo_folds_data:
        X_tr_df, y_tr = X_full_df.iloc[train_idx].copy(), y[train_idx]
        X_va_df, y_va = X_full_df.iloc[val_idx].copy(), y[val_idx]
        
        imp = SimpleImputer(strategy='mean')
        X_tr_imp = imp.fit_transform(X_tr_df[fold_feats])
        X_va_imp = imp.transform(X_va_df[fold_feats])
        
        model.fit(X_tr_imp, y_tr)
        logo_preds[val_idx] = model.predict(X_va_imp)
        
    logo_mae = mean_absolute_error(y, logo_preds)
    logo_rmse = np.sqrt(mean_squared_error(y, logo_preds))
    logo_r2 = r2_score(y, logo_preds)
    
    strict_cv_results.append({
        'Algorithm': model_name,
        '5F-KF MAE': f"{np.mean(r_maes):.2f}",
        '5F-KF RMSE': f"{np.mean(r_rmses):.2f}",
        '5F-KF R^2': f"{np.mean(r_r2s):.3f}",
        '5F-GKF MAE': f"{np.mean(g_maes):.2f}",
        '5F-GKF RMSE': f"{np.mean(g_rmses):.2f}",
        '5F-GKF R^2': f"{np.mean(g_r2s):.3f}",
        'LOOCV MAE': f"{loo_mae:.2f}",
        'LOOCV RMSE': f"{loo_rmse:.2f}",
        'LOOCV R^2': f"{loo_r2:.3f}",
        'LOGOCV MAE': f"{logo_mae:.2f}",
        'LOGOCV RMSE': f"{logo_rmse:.2f}",
        'LOGOCV R^2': f"{logo_r2:.3f}"
    })

strict_summary_df = pd.DataFrame(strict_cv_results)
print("\n=== Scale-Invariant Cross-Validation Evaluation Summary ===")
display(strict_summary_df)

=== Computing Scale-Invariant Cross-Validation (146 30s Records) ===

=== Benchmarking Scale-Invariant Cross-Validation Across All 4 Methods (146 Records) ===


=== Scale-Invariant Cross-Validation Evaluation Summary ===


,Algorithm,5F-KF MAE,5F-KF RMSE,5F-KF R^2,5F-GKF MAE,5F-GKF RMSE,5F-GKF R^2,LOOCV MAE,LOOCV RMSE,LOOCV R^2,LOGOCV MAE,LOGOCV RMSE,LOGOCV R^2
0,Random Forest,44.42,56.76,-0.138,46.25,57.55,-0.354,43.80,56.80,-0.075,45.23,57.98,-0.120
1,Gradient Boosting,44.44,58.15,-0.221,49.80,62.73,-0.633,42.61,57.41,-0.098,47.57,60.95,-0.238
2,Extra Trees,43.59,54.56,-0.036,45.14,56.01,-0.266,42.99,54.79,-0.000,44.70,56.77,-0.074
3,XGBoost,46.05,59.79,-0.308,48.85,61.97,-0.587,43.35,58.10,-0.125,46.52,60.07,-0.202
4,Ridge Regression,43.50,55.93,-0.089,45.36,57.08,-0.301,43.17,55.54,-0.028,43.81,57.01,-0.083
5,CatBoost,44.02,56.29,-0.122,45.73,57.09,-0.310,42.50,55.13,-0.013,43.96,56.45,-0.062
6,LightGBM,45.53,58.52,-0.163,50.27,62.80,-0.594,44.37,58.84,-0.154,46.64,60.77,-0.230


# Final Model Training & Evaluation on Scale-Invariant 30s Dataset ($N = 146$)

After validating cross-validation performance, feature selection and model training are fitted on the **full 146-segment 30s dataset** to produce final deployment models.

In [ ]:
# Select final top 10 features on full dataset
final_selected_features = select_features_on_fold_train(X_full_df, y, all_ppg_feature_names)
print(f"✓ Final Selected Scale-Invariant Features ({len(final_selected_features)}): {final_selected_features}")

imp_final = SimpleImputer(strategy='mean')
X_final_imp = imp_final.fit_transform(X_full_df[final_selected_features])

full_dataset_results = []
trained_final_models = {}

for model_name, model in models.items():
    model.fit(X_final_imp, y)
    trained_final_models[model_name] = model
    
    preds = model.predict(X_final_imp)
    tr_mae = mean_absolute_error(y, preds)
    tr_rmse = np.sqrt(mean_squared_error(y, preds))
    tr_r2 = r2_score(y, preds)
    
    full_dataset_results.append({
        'Algorithm': model_name,
        'Full Dataset MAE (mg/dL)': f"{tr_mae:.2f}",
        'Full Dataset RMSE (mg/dL)': f"{tr_rmse:.2f}",
        'Full Dataset R^2': f"{tr_r2:.3f}"
    })

full_summary_df = pd.DataFrame(full_dataset_results)
print("\n=== Final Model Training Results (146 Scale-Invariant 30s PPG Dataset) ===")
display(full_summary_df)

# Plot Actual vs Predicted BGL for Best Model (CatBoost / Gradient Boosting)
best_model_name = 'CatBoost'
best_preds = trained_final_models[best_model_name].predict(X_final_imp)

plt.figure(figsize=(8, 6))
plt.scatter(y, best_preds, alpha=0.7, color='#2563eb', edgecolors='k', s=60, label='30s Scale-Invariant Segments')
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2, label='Ideal 1:1 Line')
plt.xlabel('Measured BGL (mg/dL)', fontsize=12)
plt.ylabel('Predicted BGL (mg/dL)', fontsize=12)
plt.title(f'Actual vs Predicted Blood Glucose ({best_model_name} on Scale-Invariant 30s Dataset)', fontsize=13)
plt.legend()
plt.tight_layout()
plt.show()